# jupyter-tts-alerts: an example run

A small stand-in for the kind of notebook this package was written for: one that runs for hours,
where you would rather be anywhere than watching it. Here the slow parts are faked with
`time.sleep`, so the whole thing finishes in about fifteen seconds.

Turn your volume up, restart the kernel and use **Run All**. Each step says what you should hear.
Speech is queued and never talks over itself, so with Run All it trails the cells by a few
seconds; in a real notebook, where cells take minutes, it keeps up. The last cell fails on
purpose, to show what a crash sounds like, so Run All still gets through every cell before it.

You need the package in the kernel's environment: `pip install jupyter-tts-alerts`, or
`pip install .` from a clone of [the repository](https://github.com/NotEoin/jupyter-tts-alerts).
On Linux you also need a speech engine (`sudo apt install espeak-ng`). For a feature-by-feature
checklist instead of a worked example, see
[`verify-alerts.ipynb`](https://github.com/NotEoin/jupyter-tts-alerts/blob/main/notebooks/verify-alerts.ipynb).

## 1. Switch it on

Importing the package is the whole setup; nothing else is required. The rest of this notebook is
optional tuning: a lower threshold, and a few `# tts:` comments to show what they do.

**You should hear "Starting notebook."** You should also see a banner naming the speech engine it
found. If you see the banner but hear nothing, step 2 shows what the package can check.

In [ ]:
import time

import jupyter_tts_alerts as tts

## 2. Set the threshold

Cells under five seconds stay quiet by default. This example lowers that to two seconds so the
whole run stays short; in a real notebook, keep the default or raise it. `status()` shows the
engine, the two voices, what will be announced, and any sign that the kernel is running on a
machine you are not sitting at.

**You should hear nothing** from this cell: it only changes a setting and prints a summary.

In [ ]:
tts.configure(min_seconds=2)  # the default is 5; lowered so this example runs quickly
tts.status()

## 3. Load the data

A slow cell is announced when it finishes, with the `In [n]` number beside it and how long it took.

**You should hear "Cell 3. 3 seconds."** (The cell numbers here assume a fresh kernel and
Run All.)

In [ ]:
def load_measurements(count):
    """Stand-in for a slow read from disk or a database."""
    time.sleep(3)
    return [i % 1000 / 10 for i in range(count)]


data = load_measurements(10_000)
len(data)

## 4. Quick looks stay quiet

**You should hear nothing.** Fast cells pass without comment, so you are not narrated at while
you work.

In [ ]:
mean = sum(data) / len(data)
min(data), max(data), mean

## 5. Train, with progress reports

`tts.say()` speaks straight away, from inside a cell that is still running. The `# tts:` comment
on the first line swaps the usual announcement for a sentence you will recognise from the next
room.

**You should hear "epoch 1 of 3 done", "epoch 2 of 3 done", "epoch 3 of 3 done", then
"training finished".**

In [ ]:
# tts: training finished
EPOCHS = 3
estimate = 0.0
for epoch in range(1, EPOCHS + 1):
    time.sleep(2)  # stand-in for one epoch of real work
    estimate += 0.5 * (mean - estimate)
    tts.say(f"epoch {epoch} of {EPOCHS} done")
round(estimate, 2)

## 6. A cell you keep re-running

`# tts: off` keeps a slow cell quiet, which suits the one you are still tweaking.

**You should hear nothing new**, although this cell takes longer than the threshold. Anything
still playing is step 5 catching up.

In [ ]:
# tts: off
time.sleep(2.5)  # stand-in for a plot you are still getting right
print(f"estimate {estimate:.2f} against a true mean of {mean:.2f}")

## 7. The end of the run

`# tts: final` also announces the total runtime, counted from the import: the number worth knowing
when you come back to a run you started hours ago. It belongs on your last cell. Here a crash
follows it, because Jupyter stops at an error, so the crash has to come last.

**You should hear "Cell 7. 2 seconds.", then "Notebook complete. Total runtime 13 seconds."**
That is give or take a second with Run All; run cell by cell, the total also counts the time you
spent reading.

In [ ]:
# tts: final
results = {"estimate": estimate, "samples": len(data)}
time.sleep(2)  # stand-in for writing the results to disk

## 8. What a crash sounds like

This cell has a bug in it on purpose. A failure is announced however fast the cell was, and in the
second voice, so you can tell a crash from a finish without catching the words. Here it waits for
step 7 to finish speaking; in a real run there would be nothing queued ahead of it.

**You should hear "Error in cell 8. Key error." in the other voice.** Jupyter stops at an error,
which is why this is the last cell.

In [ ]:
results["accuracy"]  # a typo-sized bug of the kind that usually surfaces hours in

---

To stop the announcements for the rest of this session, run `tts.disable()`. To keep the package
quiet in every notebook without editing any of them, set `JUPYTER_TTS_ENABLED=0` in the
environment. Every setting is described in the
[README](https://github.com/NotEoin/jupyter-tts-alerts#readme).